In [2]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Reducimos el uso de hilos para evitar que el kernel se bloquee en Windows.
torch.set_num_threads(1)

# 1. Carga y preprocesamiento con Pandas (igual que en el Lab 2 original)
# IMPORTANTE: Asegúrate de que la ruta coincida con tu estructura de carpetas

df = pd.read_csv('datasets/weatherAUS.csv') 
df = df.drop(['Date', 'Location'], axis=1)
df = df.dropna()
df = pd.get_dummies(df, drop_first=True)
df = df.astype(float)

X_pandas = df.drop('Temp3pm', axis=1).values
y_pandas = df['Temp3pm'].values

# División 80/20
m = len(y_pandas)
train_size = int(m * 0.8)
np.random.seed(42)
indices = np.random.permutation(m)
X, y = X_pandas[indices], y_pandas[indices]

X_train, y_train = X[:train_size], y[:train_size]
X_test, y_test = X[train_size:], y[train_size:]

# Normalización
def featureNormalize(X_train, X_test):
    mu = np.mean(X_train, axis=0)
    sigma = np.std(X_train, axis=0)
    sigma[sigma == 0] = 1
    X_train_norm = (X_train - mu) / sigma
    X_test_norm = (X_test - mu) / sigma
    return X_train_norm, X_test_norm

X_train_norm, X_test_norm = featureNormalize(X_train, X_test)

# 2. Creación de la clase Dataset de PyTorch 
class CustomDataset(Dataset):
    def __init__(self, X, y):
        # Convertimos los arreglos de NumPy a Tensores de PyTorch
        self.X = torch.tensor(X, dtype=torch.float32)
        # Para regresión (predecir un número), y debe tener forma [n, 1] y ser float32
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# 3. Creación de los objetos Dataset
train_dataset = CustomDataset(X_train_norm, y_train)
test_dataset = CustomDataset(X_test_norm, y_test)

# 4. Creación de los DataLoaders
# num_workers=0 evita procesos secundarios que pueden tumbar el kernel en Windows.
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Número de características (entradas): {X_train_norm.shape[1]}")
print(f"Total de lotes (batches) de entrenamiento: {len(train_loader)}")

Número de características (entradas): 62
Total de lotes (batches) de entrenamiento: 706


## Modelo Costo y Optimizador

In [3]:
import torch.nn as nn
import torch.optim as optim

# Definimos la arquitectura del modelo (Regresión Lineal = 1 capa)
class LinearRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super(LinearRegressionModel, self).__init__()
        # 62 entradas -> 1 salida (Temp3pm)
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x)

# Instanciamos el modelo
input_dim = X_train_norm.shape[1]
model = LinearRegressionModel(input_dim)

# Función de Costo (Mean Squared Error) y Optimizador
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

print(model)

LinearRegressionModel(
  (linear): Linear(in_features=62, out_features=1, bias=True)
)


##  Bucle de Entrenamiento y Guardado de Pesos
Procesamos los datos por lotes iterativamente. En cada paso: pasamos los datos (forward), calculamos el error, calculamos gradientes (backward) y actualizamos los parámetros

In [4]:
num_epochs = 50
train_losses = []

print("Iniciando entrenamiento con PyTorch...")
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0

    for X_batch, y_batch in train_loader:
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)

    if (epoch + 1) % 10 == 0:
        print(f"Época [{epoch+1}/{num_epochs}], Costo (MSE): {avg_loss:.4f}")

# Guardar los pesos del modelo
torch.save(model.state_dict(), 'mejor_modelo_lab2.pt')
print("\n¡Pesos del modelo guardados exitosamente como 'mejor_modelo_lab2.pt'!")

# Crear la gráfica sin ejecutar Matplotlib, que bloquea este kernel.
from PIL import Image, ImageDraw

width, height = 800, 500
margin = 60
image = Image.new('RGB', (width, height), 'white')
draw = ImageDraw.Draw(image)
draw.line((margin, margin, margin, height - margin), fill='black', width=2)
draw.line((margin, height - margin, width - margin, height - margin), fill='black', width=2)

loss_min = min(train_losses)
loss_max = max(train_losses)
loss_range = max(loss_max - loss_min, 1e-8)
points = []
for index, loss_value in enumerate(train_losses):
    x = margin + index * (width - 2 * margin) / max(len(train_losses) - 1, 1)
    y = height - margin - (loss_value - loss_min) * (height - 2 * margin) / loss_range
    points.append((int(x), int(y)))

if len(points) > 1:
    draw.line(points, fill='green', width=3)

draw.text((margin, 15), 'Regresion Lineal en PyTorch: Reduccion del Costo', fill='black')
draw.text((margin, height - margin + 15), 'Epocas', fill='black')
draw.text((5, margin - 20), 'MSE', fill='black')
image.save('costo_entrenamiento_lab2.png')
print("Grafica guardada exitosamente como 'costo_entrenamiento_lab2.png'")

Iniciando entrenamiento con PyTorch...
Época [10/50], Costo (MSE): 0.8291
Época [20/50], Costo (MSE): 0.8290
Época [30/50], Costo (MSE): 0.8287
Época [40/50], Costo (MSE): 0.8296
Época [50/50], Costo (MSE): 0.8293

¡Pesos del modelo guardados exitosamente como 'mejor_modelo_lab2.pt'!
Grafica guardada exitosamente como 'costo_entrenamiento_lab2.png'
